In [1]:
from torchvision.datasets import CelebA
from torchvision import transforms

from torchvision.models import ResNet18_Weights

import torchvision
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import random_split


from torchmetrics.classification import BinaryAUROC, Accuracy

import torch
from pytorch_lightning.loggers import WandbLogger

import numpy as np 

import wandb

from datetime import datetime
from tqdm.notebook import tqdm

from collections import OrderedDict

from xaikd import utils, attributors, models

import pandas as pd
from xaikd.bases import PRCAReconGreedy

from matplotlib import pyplot as plt 

import numpy.typing as npt

import os

In [2]:
DATA_ROOT = "../../datasets"

WANDB_PROJECT = "xaikd-training-teacher-models"
WANDB_GROUP = "celeba"

NUM_WORKERS = 16
BATCH_SIZE = 64
NUM_ATTRIBUTES = 40

TRAINING_SIZE = 0.01

SEED = 1

DEVICE = "cuda"

ARCH = "resnet18"

RUN_ID = "n8r0q2vb"
# RUN_ID = "6oj5aaxl" # imagenet pretrained
# RUN_ID = "dskgwbyk" # fc.bias= False


# WANDB_PROJECT = "kitchen-sink"
# RUN_ID = "jtp7uv29"

In [3]:
TRANSFORMATION_DEFAULT = ResNet18_Weights.IMAGENET1K_V1.transforms()

In [4]:
TRANSFORMATION_DEFAULT.mean, TRANSFORMATION_DEFAULT.std

([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

In [5]:
ds_train = CelebA(
    root=DATA_ROOT, split="train", target_type="attr",
    transform=TRANSFORMATION_DEFAULT
)

trng = torch.Generator()
trng.manual_seed(1)
ds_train, _ = random_split(ds_train, [TRAINING_SIZE, 1-TRAINING_SIZE], generator=trng)

dl_train = DataLoader(
    ds_train,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,    
    shuffle=False,
)

In [6]:
ds_val = CelebA(
    root=DATA_ROOT, split="valid", target_type="attr",
    transform=TRANSFORMATION_DEFAULT
)

ds_val, _ = random_split(ds_val, [TRAINING_SIZE, 1-TRAINING_SIZE], generator=trng)

dl_val = DataLoader(
    ds_val,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,    
    shuffle=False,
)

In [7]:
class MultiTaskHead(nn.Module):
    def __init__(self, in_dims, num_tasks, out_per_task):
        super().__init__()
        
        arr_heads = []
        for tix in range(num_tasks):
            head = nn.Linear(in_features=in_dims, out_features=out_per_task)
            arr_heads.append(head)
        self.arr_heads = nn.ModuleList(arr_heads)
        self.task_id = None

    def forward(self, x):
        if self.task_id is not None:
            return self.arr_heads[self.task_id](x)
    
        arr_out = []

        
        for head in self.arr_heads:
            headout = head(x)
            b, d = headout.shape
            arr_out.append(headout.reshape(b, 1, d))

        out = torch.cat(arr_out, dim=1)
        return out
        
def get_state_dict(run_id):
    agent = wandb.Api()

    artifact: wandb.Artifact = agent.artifact(
        f"{WANDB_PROJECT}/model-{RUN_ID}:latest"
    )

    artifact_dir = artifact.download(root="/tmp")

    ckpt = torch.load(
        f"{artifact_dir}/model.ckpt",
        map_location=torch.device("cpu"),
        weights_only=False,
    )

    state_dict = ckpt["state_dict"]

    new_dict = OrderedDict()
    for k, v in state_dict.items():
        new_k = k.replace("encoder.", "")
        new_dict[new_k] = v
        
    return new_dict

def get_model(arch):

    state_dict = get_state_dict(RUN_ID)
    
    if arch == "resnet18":
        model = torchvision.models.resnet18(weights=None, num_classes=NUM_ATTRIBUTES)
        # out_dims, in_dims = model.fc.weight.shape
        
        # model.fc = MultiTaskHead(in_dims, NUM_ATTRIBUTES, 2)
    
    model.load_state_dict(state_dict)

    model.eval()
    model.to(DEVICE)
    
    return model
    
model = get_model(ARCH);

wandb: Downloading large artifact model-n8r0q2vb:latest, 128.27MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:6.1


In [8]:
def estimate_task_performance(model, dl, task_id, verbose=False):

    # metric = Accuracy(task="multiclass", num_classes=2)
    metric = BinaryAUROC(thresholds=20)
    model.to(DEVICE)

    for x, y in tqdm(dl, disable=not verbose):
        x = x.to(DEVICE)
        task_logit = model(x)[:, task_id].cpu()
        task_target = y[:, task_id]
        metric.update(task_logit, task_target)

    metric = float(metric.compute())
    metric = np.max([metric, 1-metric])
    return metric

# cross-check with wandb that we have correct results
estimate_task_performance(model, dl_val, task_id=0, verbose=True)

  0%|          | 0/4 [00:00<?, ?it/s]

0.9554470181465149

# Extract Activation and Context Vectors for Task

In [9]:
raise NotImplementedError()

NotImplementedError: 

In [37]:
class VoidAttributor:

    def __enter__(self, **kwargs):
        pass

    def __exit__(self, type, value, tb):
        pass

def compute_logodd_winning(logits):

    return torch.sign(logits).detach() * logits

def compute_logits(logits):

    return logits

class OutputQuantity:
    def __call__(self, logits, target_logits):
        raise NotImplementedError()
    def __str__(self):
        return self.__class__.__name__
        
class LogOddWinningClass(OutputQuantity):
    def __call__(self, logits, targets):
        return torch.sign(logits).detach() * logits


class LogOddSquared(OutputQuantity):
    def __call__(self, logits, targets):
        return (torch.sign(logits).detach() * logits).pow(2)


        
class LogOddPositiveClass(OutputQuantity):
    def __call__(self, logits, targets):
        return logits


class LogOddTargetClass(OutputQuantity):
    def __call__(self, logits, targets):
        return torch.sign(2*targets-1) * logits
        

class BinaryCrossEntropyWinning(OutputQuantity):
    def __call__(self, logits, targets):
        winning_target = torch.sign(logits).detach()
        return - F.binary_cross_entropy_with_logits(logits, winning_target)

def extract_activation_context_for_task(
    model: nn.Module,
    layer: str,
    data_loader: DataLoader,
    task_id: int,
    output_quantity: OutputQuantity,
    use_lrp=True,
    seed=1,
    device=DEVICE,
    number_of_selected_spatial_locations=20,
    strict_mode=False,
    verbose=False
):

    arr_logodds = []
    arr_act = []
    arr_ctx = []

    rng = np.random.default_rng(seed=1)

    task_query_vector = F.one_hot(torch.tensor(task_id), num_classes=NUM_ATTRIBUTES).to(DEVICE)
    try:
        model.fc.task_id = task_id
        
        module, hook = utils.interceptor.attach_hook_intercept_layer_output(
            model, layer, should_retain_grad=True, detach_output=False
        )

        attributor = attributors.make_attributor_for(
            model,
            (
                TRANSFORMATION_DEFAULT.mean, TRANSFORMATION_DEFAULT.std
            )
        ) if use_lrp else VoidAttributor()
        
        with attributor:
            for batch in tqdm(data_loader, desc=f"[layer={layer}; use_lrp={use_lrp}] extract act ctx (wrt {output_quantity})"):
                x, y = batch
                x = x.to(device)

                if use_lrp:
                    raise NotImplementedError("obsolete")
                    _ = attributor.forward(x, lambda logits: logits * task_query_vector)
    
                    act = utils.interceptor.get_output(module)
    
                    assert act.grad is not None
                    rel = act.grad
    
    
                    ctx = torch.where(act.abs() > 0, rel / act, 0)
                    assert torch.isfinite(ctx).all()
                    
                    if strict_mode:
                        np.testing.assert_allclose(
                            (act * ctx).detach().cpu().numpy(),
                            rel.detach().cpu().numpy(),
                            atol=1e-6,
                        )
                else:
                    logits = model(x)
                    task_logit = logits[:, task_id]
                    task_target = y[:, task_id].to(device)
                    
                    quantities = output_quantity(task_logit, task_target)
                    # quantities = compute_logits(task_logit)
                    
                    (quantities).sum().backward()
                    act = utils.interceptor.get_output(module)
    
                    assert act.grad is not None
                    ctx = act.grad
                
                output_dimensions = act.shape[1:]


                assert ctx.shape == act.shape

                # act = act.detach().cpu().numpy()
                # ctx = ctx.detach().cpu().numpy()

                # selected_act, selected_ctx = utils.subsample_tensors(
                #     act,
                #     ctx,
                #     num_locations=number_of_selected_spatial_locations,
                #     rng=rng,
                # )
                
                selected_act = act.detach().cpu().numpy()
                selected_ctx = ctx.detach().cpu().numpy()
                arr_act.append(selected_act)
                arr_ctx.append(selected_ctx)
                arr_logodds.append(quantities.detach().cpu().numpy())

    finally:
        hook.remove()
        model.fc.task_id = None

    print(f"{layer}: output-dims={output_dimensions}")

    arr_act = np.vstack(arr_act)
    arr_ctx = np.vstack(arr_ctx)
    # print("arr_act.shape", arr_act.shape)
    arr_logodds = np.hstack(arr_logodds)
    # print("arr_logodd.shape", arr_logodds.shape)

    return arr_logodds, arr_act, arr_ctx

def ano():

    for use_lrp in [False]:

        for output_quantity in [
            LogOddSquared(),
            # BinaryCrossEntropyWinning(),
            # LogOddWinningClass(),
            # LogOddTargetClass(),
            # LogOddPositiveClass(),
        ]:
        
            extract_activation_context_for_task(
                model, 
                "layer3",
                dl_train,
                task_id=25,
                output_quantity=output_quantity,
                use_lrp=use_lrp
            )
        
        print(f"Sanity check: [use_lrp={use_lrp}; output_quantity={output_quantity}] passed!")
ano()

[layer=layer3; use_lrp=False] extract act ctx (wrt LogOddSquared):   0%|          | 0/26 [00:00<?, ?it/s]

layer3: output-dims=torch.Size([256, 14, 14])
Sanity check: [use_lrp=False; output_quantity=LogOddSquared] passed!


# Constructing Basis 

In [19]:
def _solve_eigvecs(cov, sort_func=lambda x: x):
    eigvals, eigvecs = np.linalg.eigh(cov)

    assert len(eigvals.shape) == 1

    indices = np.argsort(-sort_func(eigvals))
    eigvals = eigvals[indices]
    eigvecs = eigvecs[:, indices]

    return eigvecs

def flatten_4d_array(x):
    b, nc, h, w = x.shape
    return np.transpose(x, (0, 2, 3, 1)).reshape((b*h*w, nc))

@torch.no_grad()
def ano():
    H = 8
    W = 7
    B = 3
    NC = 5
    x = np.random.randn(B, NC, H, W)

    x_flat = flatten_4d_array(x) 

    for cix, (i, h, w) in enumerate([
        (0, 0, 0),
        (0, 0, 2),
        (0, 1, 0),
        (1, 0, 0),
        (2, 0, 0),
        (2, 1, 0),
    ]):
        ii = i * (H * W) + h * W + w
            
        np.testing.assert_allclose(
            x_flat[ii],     
            x[i, :, h,w], 
        )
        print(f"Passed: Case {cix}!")
ano()

Passed: Case 0!
Passed: Case 1!
Passed: Case 2!
Passed: Case 3!
Passed: Case 4!
Passed: Case 5!


In [20]:
class BasisInterface:
    def get_Uk(self, k: int) -> npt.NDArray:
        raise NotImplementedError()   

class PCA(BasisInterface):
    def __init__(self, arr_logodds, arr_act, arr_ctx, layer=None, task_id=None):
        arr_act = flatten_4d_array(arr_act)
        arr_ctx = flatten_4d_array(arr_ctx)
        cov = arr_act.T @ arr_act
        self.U = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]

class GradPCA(BasisInterface):
    def __init__(self, arr_logodds, arr_act, arr_ctx, layer=None, task_id=None):
        arr_act = flatten_4d_array(arr_act)
        arr_ctx = flatten_4d_array(arr_ctx)
        
        cov = arr_ctx.T @ arr_ctx
        self.U = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]


## PRCA Variants

In [50]:
class PRCA(BasisInterface):
    def __init__(self, arr_logodds, arr_act, arr_ctx, layer=None, task_id=None):
        arr_act = flatten_4d_array(arr_act)
        arr_ctx = flatten_4d_array(arr_ctx)
        
        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        self.U = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]

class PRCASortAbs(BasisInterface):
    def __init__(self, arr_logodds, arr_act, arr_ctx, layer=None, task_id=None):
        arr_act = flatten_4d_array(arr_act)
        arr_ctx = flatten_4d_array(arr_ctx)
        
        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        self.U = _solve_eigvecs(cov, sort_func=np.abs)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]


class PRCAScaled(BasisInterface):
    def __init__(self, arr_logodds, arr_act, arr_ctx, layer=None, task_id=None):
        arr_act = flatten_4d_array(arr_act)
        arr_ctx = flatten_4d_array(arr_ctx)

        arr_factors = ((arr_act)*(arr_ctx)).sum(axis=1, keepdims=True)
        
        arr_act = arr_act * arr_factors
        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        self.U = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]


class PRCAScaledSortAbs(BasisInterface):
    def __init__(self, arr_logodds, arr_act, arr_ctx, layer=None, task_id=None):
        arr_act = flatten_4d_array(arr_act)
        arr_ctx = flatten_4d_array(arr_ctx)

        arr_factors = ((arr_act)*(arr_ctx)).sum(axis=1, keepdims=True)
        
        arr_act = arr_act * arr_factors
        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        self.U = _solve_eigvecs(cov, sort_func=np.abs)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]
        
class PRCAScaledSubtractMedianSortAbs(BasisInterface):
    def __init__(self, arr_logodds, arr_act, arr_ctx, layer=None, task_id=None):
        arr_act = flatten_4d_array(arr_act)
        arr_ctx = flatten_4d_array(arr_ctx)

        arr_factors = ((arr_act)*(arr_ctx)).sum(axis=1, keepdims=True)
        arr_factors =  2*arr_factors - np.median(arr_factors)
        
        arr_act = arr_act  * arr_factors
        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        self.U = _solve_eigvecs(cov, sort_func=np.abs)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]

In [51]:
class PRCAReconNonGreedyLearner:
    def __init__(self, task_id, layer):
        self.task_id = task_id
        self.layer = layer
    def fit(
        self, arr_logodds, arr_act: npt.NDArray, arr_ctx: npt.NDArray, k: int,  U_init=None, device="cpu",
    ) -> npt.NDArray:
        n, d, _, _ = arr_act.shape

        assert arr_ctx.shape == arr_act.shape, arr_ctx.shape
        
        lr = 1e-4
        epochs = 5000
                
        n, d, w, h  = arr_act.shape

        scale_act = ((np.mean(arr_act**2) ** (1 / 2)) * (d ** (1 / 4)))
        scale_ctx = ((np.mean(arr_ctx**2) ** (1 / 2)) * (d ** (1 / 4)))
        arr_act = arr_act / scale_act
        arr_ctx = arr_ctx / scale_ctx
        
        arr_act: torch.Tensor = torch.from_numpy(arr_act).to(device)
        arr_ctx: torch.Tensor = torch.from_numpy(arr_ctx).to(device)


        linear_layer = torch.nn.Linear(k, d, bias=False)
        trng = torch.Generator()
        trng.manual_seed(1)
        if U_init is None:
            U_init = torch.randn((k, d), generator=trng)
        else:
            U_init = torch.from_numpy(U_init.T)
            
        linear_layer.weight = torch.nn.Parameter(U_init)
    
        ortho_layer = torch.nn.utils.parametrizations.orthogonal(linear_layer).to(device)
        assert ortho_layer.weight.shape == (k, d)
        
        optimizer = torch.optim.Adam(ortho_layer.parameters(), lr=lr)

        rel = (arr_act*arr_ctx).sum(dim=1)
        
        pgb = tqdm(range(epochs), desc=f"{self.__class__.__name__} (k={k})")
        for epoch in pgb:
            optimizer.zero_grad()
            
            # shape: (k, d)
            U = ortho_layer.weight

            U = U.T @ U
            U = U.unsqueeze(2).unsqueeze(3)
            
            act_proj = F.conv2d(arr_act, U)

            ctx_proj = F.conv2d(arr_ctx, U)
            
            # assert act_proj.shape == (n, k, w, h), act_proj.shape
            rel_recon = (act_proj * ctx_proj).sum(dim=1)


            # shape = (n, w, h)
            loss = (rel - rel_recon).pow(2)
    
            loss = loss.mean()
            
            loss.backward()
        
            optimizer.step()

            loss = loss.detach().cpu().numpy()
                
            pgb.set_description_str(f"{self.__class__.__name__} (k={k}; lr={lr}) loss={loss:.4e}")
            
        U =  ortho_layer.weight.T.detach().cpu().numpy()
        
        # sanity_check
        np.testing.assert_allclose(U.T @ U, np.eye(k), atol=1e-4)

        return U


class PRCARecon(BasisInterface):
    def __init__(self, arr_logodds, arr_act, arr_ctx, layer=None, task_id=None):
        self.arr_act = arr_act
        self.arr_ctx = arr_ctx
        self.arr_logodds = arr_logodds
        self.layer = layer
        self.task_id = task_id
        self.is_slow = True
        
    def get_Uk(self, k: int):

        return PRCAReconNonGreedyLearner(
            layer=self.layer,
            task_id=self.task_id
        ).fit(
            self.arr_logodds, self.arr_act, self.arr_ctx, 
            k=k,
            U_init=PRCASortAbs(arr_logodds=self.arr_logodds, arr_act=self.arr_act, arr_ctx=self.arr_ctx).get_Uk(k),
            device=DEVICE
        )

# Estimating AUROCs

In [55]:
def construct_fh(Uk):
    def fh(mod, inp, outp):
        return F.conv2d(
            outp,
            (Uk@Uk.T).unsqueeze(2).unsqueeze(3)
        )
    return fh

def compute_task_aurocs_at_k(
    model, layer, task_id, 
    arr_ks,
    arr_ks_for_slow_learners,
    use_lrp,
    output_quantity,
    arr_basis_names=[
        PCA
    ],
    
    base_output_dir=f"./artifacts/experiment-basis-comparisons/celeba-{RUN_ID}"
):
    arr_logodds, arr_act, arr_ctx = extract_activation_context_for_task(
        model=model, 
        layer=layer, 
        data_loader=dl_train, 
        task_id=task_id,
        output_quantity=output_quantity,
        use_lrp=use_lrp
    )
    
    rel = (arr_act  * arr_ctx).sum(axis=1)

    module = getattr(model, layer)
    arr_ks = sorted(list(set(arr_ks + arr_ks_for_slow_learners)))

    suffix = "relevance-lrp" if use_lrp else "relevance-grad"
    output_path = f"{base_output_dir}/task-{task_id}/{layer}/{suffix}"
    os.makedirs(output_path, exist_ok=True)

    arr_dfs = []
    for basis_class in arr_basis_names:
        arr_stat_rows = []

        basis: BasisInterface = basis_class(
            arr_logodds=arr_logodds,
            arr_act=arr_act, 
            arr_ctx=arr_ctx,
            layer=layer,
            task_id=task_id
        )
        basis_name = basis_class.__name__

        for k in tqdm(
            arr_ks_for_slow_learners if hasattr(basis, "is_slow") else arr_ks, 
            desc=f"[{basis_name:<20s}] Estimating Performance"
        ):
        
            if ("-k" in basis_name) and not f"{k}" == basis_name.split("k")[1]:
                continue

            Uk = basis.get_Uk(k=k)

        
            # arr_recon_act = (arr_act @ Uk) @ Uk.T

            
            # projected_rel = ((arr_act @ Uk) * (arr_ctx @ Uk)).sum(axis=1)
            
            # assert rel.shape == projected_rel.shape == (rel.shape[0], )
            
            # recon_err = np.linalg.norm(arr_act - arr_recon_act, axis=1).mean()
            # rel_recon_err = ((rel - projected_rel) **2).mean()
            # perc_sign_align = (np.sign(rel) * np.sign(projected_rel)).mean()

            row = dict(
                
                task_id=task_id,
                layer=layer,
                use_lrp=use_lrp,
                output_quantity=f"{output_quantity}",
                k=k,
                basis_name=basis_name,
                # recon_err=recon_err,
                # rel_recon_err=rel_recon_err,
                # perc_sign_align=perc_sign_align,
            )

            Uk = torch.from_numpy(Uk).to(DEVICE)

            try:
                hook = module.register_forward_hook(construct_fh(Uk))
                
                for label, dl in [
                    # ("train", dl_train),
                    ("val", dl_val)
                ]:
                    row[f"{label}_auroc"] = estimate_task_performance(model, dl, task_id)
                
            finally:
                hook.remove()

            if basis_class == PRCARecon:
                print(f"k={k}", row)
            arr_stat_rows.append(row)
            
     
        df = pd.DataFrame(arr_stat_rows)
        df.to_csv(
            f"{output_path}/{basis_name}.csv",
            index=False
        )

        arr_dfs.append(df)

    df = pd.concat(arr_dfs).sort_values(by=["k", f"val_auroc"], ascending=[True, False])
    print(f"Checking results at {output_path}")
    return df

compute_task_aurocs_at_k(
    model, layer="layer3", task_id=0, 
    arr_basis_names=[
        PCA,
        GradPCA,
        
        # PRCA,     
        # PRCASortAbs,
        
        # PRCAScaled,
        # PRCAScaledSortAbs,
        # PRCAScaledSubtractMedianSortAbs,

        # PRCARecon
    ],
    arr_ks=[5, 10, 20, 30, 256],
    arr_ks_for_slow_learners=[5, 10, 20],
    use_lrp=False,
    output_quantity=LogOddWinningClass(),
    base_output_dir="./tmp/celeba"
)

[layer=layer3; use_lrp=False] extract act ctx (wrt LogOddWinningClass):   0%|          | 0/26 [00:00<?, ?it/s]

layer3: output-dims=torch.Size([256, 14, 14])


[PCA                 ] Estimating Performance:   0%|          | 0/5 [00:00<?, ?it/s]

[GradPCA             ] Estimating Performance:   0%|          | 0/5 [00:00<?, ?it/s]

Checking results at ./tmp/celeba/task-0/layer3/relevance-grad


,task_id,layer,use_lrp,output_quantity,k,basis_name,val_auroc
0,0,layer3,False,LogOddWinningClass,5,GradPCA,0.900978
0,0,layer3,False,LogOddWinningClass,5,PCA,0.580168
1,0,layer3,False,LogOddWinningClass,10,GradPCA,0.945670
1,0,layer3,False,LogOddWinningClass,10,PCA,0.802235
2,0,layer3,False,LogOddWinningClass,20,GradPCA,0.969134
2,0,layer3,False,LogOddWinningClass,20,PCA,0.893017
3,0,layer3,False,LogOddWinningClass,30,GradPCA,0.980587
3,0,layer3,False,LogOddWinningClass,30,PCA,0.948883
4,0,layer3,False,LogOddWinningClass,256,PCA,0.955447
4,0,layer3,False,LogOddWinningClass,256,GradPCA,0.955447


# Getting Results

In [56]:
ARR_LAYER_DIMENSIONS = utils.get_dimensions_at_layers(
    model=model,
    dataloader=dl_train,
    layers=["layer1", "layer2", "layer3", "layer4"],
    device=DEVICE
)

In [57]:
ARR_LAYER_DIMENSIONS

{'layer1': 64, 'layer2': 128, 'layer3': 256, 'layer4': 512}

In [ ]:
for use_lrp in [False]:
    
    for layer in tqdm(["layer1", "layer2", "layer3", "layer4"], desc=f"use_lrp={use_lrp}"):
        d = ARR_LAYER_DIMENSIONS[layer]
        
        arr_ks = sorted(set(
            np.arange(1, 10).tolist() + 
            np.linspace(1, d, 8).astype(int).tolist()
        ))


        for task_id in [0, 25]:
                
            compute_task_aurocs_at_k(
                model, layer=layer, task_id=task_id, 
                arr_basis_names=[
                    PCA, GradPCA,
                    
                    PRCA,     
                    PRCASortAbs,
                    
                    PRCAScaled,
                    PRCAScaledSortAbs,
                    PRCAScaledSubtractMedianSortAbs,
            
                    PRCARecon,
                ],
                arr_ks=arr_ks,
                arr_ks_for_slow_learners=arr_ks,
                output_quantity=LogOddWinningClass(),
                use_lrp=use_lrp,
            )

use_lrp=False:   0%|          | 0/4 [00:00<?, ?it/s]

[layer=layer1; use_lrp=False] extract act ctx (wrt LogOddWinningClass):   0%|          | 0/26 [00:00<?, ?it/s]

layer1: output-dims=torch.Size([64, 56, 56])


[PCA                 ] Estimating Performance:   0%|          | 0/16 [00:00<?, ?it/s]

[GradPCA             ] Estimating Performance:   0%|          | 0/16 [00:00<?, ?it/s]

...

PRCAReconNonGreedyLearner (k=110):   0%|          | 0/5000 [00:00<?, ?it/s]

k=110 {'task_id': 25, 'layer': 'layer3', 'use_lrp': False, 'output_quantity': 'LogOddWinningClass', 'k': 110, 'basis_name': 'PRCARecon', 'val_auroc': 0.7006052732467651}


PRCAReconNonGreedyLearner (k=146):   0%|          | 0/5000 [00:00<?, ?it/s]

k=146 {'task_id': 25, 'layer': 'layer3', 'use_lrp': False, 'output_quantity': 'LogOddWinningClass', 'k': 146, 'basis_name': 'PRCARecon', 'val_auroc': 0.7018765211105347}


PRCAReconNonGreedyLearner (k=183):   0%|          | 0/5000 [00:00<?, ?it/s]

In [ ]:
print(f"finished at {datetime.now()}")